# Entity NER Agenda Graph — Per-Type Gephi Export

Reads the three-sheet review workbook and produces **three independent bipartite
graph CSV pairs** — one per entity type (PER / ORG / LOC).

Each graph: all source sub-units ↔ type-T entities.  
Edge weight = coverage share (distinct docs in sub-unit mentioning entity /
total docs in sub-unit). Weights do **not** sum to 1 per sub-unit.

Output files: `per_nodes.csv`, `per_edges.csv`, `org_nodes.csv`, `org_edges.csv`,
`loc_nodes.csv`, `loc_edges.csv`.

In [8]:
import ast
import re
import pathlib
import pandas as pd

NB_DIR = pathlib.Path(".").resolve()

# ============================================================
# PARAMETERS
# ============================================================

REVIEW_XLSX = "entity_review.xlsx"   # three-sheet workbook (PER / ORG / LOC)

SOURCES = {
    "news": {
        "path":        "../../news/analysis/df_with_NER.csv",
        "subunit_col": "outlet",
        "node_type":   "news",
    },
    "talkshows": {
        "path":        "../../subtitles/analysis/subs_with_NER.csv",
        "subunit_col": "program",
        "node_type":   "talkshows",
    },
    "kamer": {
        "path":        "../../tweede_kamer/analysis/Tweede_Kamer_with_NER.csv",
        "subunit_col": "type",
        "node_type":   "kamer",
    },
}

ENTITY_COLS = ["persons", "orgs", "countries"]   # all parsed per document

MIN_DOCS        = 0    # drop sub-units with fewer total documents
MIN_EDGE_WEIGHT = 0.0  # drop edges below this coverage share
# ============================================================

## 1. Helpers

In [15]:
def parse_entity_list(cell):
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if s in ("", "[]", "nan"):
        return []
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return [str(x) for x in result]
        return [str(result)]
    except (ValueError, SyntaxError):
        s = re.sub(r"^[\[\(]|[\]\)]$", "", s)
        return [x.strip().strip("'\"" ) for x in s.split(",") if x.strip()]


def normalise(s):
    return re.sub(r"\s+", " ", str(s).strip())


def slugify(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")

## 2. Load review sheets — build per-type entity lookups

Each sheet produces two dicts:
- `lookup`: normalised raw entity (lowercase) → canonical label
- `canon_set`: set of distinct canonical labels for that type

If the same normalised key appears in multiple sheets (NER misclassification),
each sheet still gets its own lookup independently — no conflict because the
graphs are built separately.

In [10]:
review_path = (NB_DIR / REVIEW_XLSX).resolve()

lookups    = {}   # sheet_type -> {norm_key: canonical}
canon_sets = {}   # sheet_type -> set of canonical labels

for sheet_type in ["PER", "ORG", "LOC"]:
    df   = pd.read_excel(review_path, sheet_name=sheet_type)
    kept = df[df["keep"].astype(str).str.strip().str.lower() == "yes"].copy()

    lookup   = {}
    canonicals = set()
    for _, row in kept.iterrows():
        key       = normalise(row["entity"]).lower()
        canonical = normalise(row["canonical"])
        lookup[key] = canonical
        canonicals.add(canonical)

    lookups[sheet_type]    = lookup
    canon_sets[sheet_type] = canonicals
    print(f"{sheet_type}: {len(kept)} keep=yes rows → {len(canonicals)} distinct canonicals "
          f"(alias mappings: {sum(1 for k,v in lookup.items() if k != v.lower())})")

# Warn about cross-sheet key collisions
all_keys = {t: set(lookups[t]) for t in lookups}
for t1, t2 in [("PER","ORG"),("PER","LOC"),("ORG","LOC")]:
    shared = all_keys[t1] & all_keys[t2]
    if shared:
        print(f"WARNING — {len(shared)} key(s) appear in both {t1} and {t2} lookups: "
              f"{sorted(shared)[:5]}{'…' if len(shared)>5 else ''}")

PER: 74 keep=yes rows → 60 distinct canonicals (alias mappings: 16)
ORG: 83 keep=yes rows → 80 distinct canonicals (alias mappings: 4)
LOC: 83 keep=yes rows → 38 distinct canonicals (alias mappings: 45)


## 3. Load source files

In [11]:
frames = {}

for arena, cfg in SOURCES.items():
    abs_path = (NB_DIR / cfg["path"]).resolve()
    if not abs_path.exists():
        print(f"[SKIP] {arena}: not found")
        continue
    df = pd.read_csv(abs_path)
    df[cfg["subunit_col"]] = (
        df[cfg["subunit_col"]].fillna("UNKNOWN").astype(str).str.strip()
    )
    frames[arena] = df
    print(f"[OK] {arena}: {len(df):,} docs, "
          f"{df[cfg['subunit_col']].nunique()} sub-units ({cfg['subunit_col']})")

[OK] news: 13,209 docs, 8 sub-units (outlet)
[OK] talkshows: 495 docs, 9 sub-units (program)
[OK] kamer: 844 docs, 3 sub-units (type)


## 4. Process all documents once

For each document, parse all entity columns, normalise, map against every
per-type lookup, and dedupe within the document. Result: one long table with
columns `(arena, subunit, doc_id, canonical, entity_type)`.

In [12]:
records = []

for arena, cfg in SOURCES.items():
    if arena not in frames:
        continue
    df          = frames[arena]
    subunit_col = cfg["subunit_col"]

    for row_idx, row in df.iterrows():
        subunit = row[subunit_col]
        doc_id  = f"{arena}:{row_idx}"

        # Collect canonicals per type (deduplicated within document)
        doc_by_type: dict[str, set] = {t: set() for t in lookups}
        for col in ENTITY_COLS:
            for raw in parse_entity_list(row.get(col)):
                norm_key = normalise(raw).lower()
                for sheet_type, lookup in lookups.items():
                    canonical = lookup.get(norm_key)
                    if canonical:
                        doc_by_type[sheet_type].add(canonical)

        for sheet_type, canonicals in doc_by_type.items():
            for canonical in canonicals:
                records.append({
                    "arena":       arena,
                    "subunit":     subunit,
                    "doc_id":      doc_id,
                    "canonical":   canonical,
                    "entity_type": sheet_type,
                })

long_df = pd.DataFrame(records)
print(f"Total (sub-unit, entity) pairings: {len(long_df):,}")
print(long_df.groupby("entity_type").size().rename("pairings").to_string())

Total (sub-unit, entity) pairings: 46,460
entity_type
LOC    20851
ORG    19051
PER     6558


## 5. Pre-compute sub-unit totals and node rows

Source sub-units are identical across all three graphs.

In [13]:
subunit_totals = {}   # (arena, subunit) -> total doc count
subunit_node_rows = []   # shared source-node records

for arena, cfg in SOURCES.items():
    if arena not in frames:
        continue
    for subunit, grp in frames[arena].groupby(cfg["subunit_col"]):
        if len(grp) < MIN_DOCS:
            continue
        subunit_totals[(arena, subunit)] = len(grp)
        subunit_node_rows.append({
            "Id":          f"src_{slugify(subunit)}",
            "Label":       subunit,
            "node_type":   cfg["node_type"],
            "entity_type": "",
            "size":        len(grp),
        })

src_nodes_df = pd.DataFrame(subunit_node_rows)
subunit_id   = dict(zip(src_nodes_df["Label"], src_nodes_df["Id"]))
print(f"{len(src_nodes_df)} source sub-unit nodes")

20 source sub-unit nodes


## 6. Build and export one graph per type

In [14]:
TYPE_PREFIX = {"PER": "per", "ORG": "org", "LOC": "loc"}

print("=" * 60)

for sheet_type in ["PER", "ORG", "LOC"]:
    prefix   = TYPE_PREFIX[sheet_type]
    type_long = long_df[long_df["entity_type"] == sheet_type]

    # --- entity nodes ---
    entity_doc_freq = (
        type_long.groupby("canonical")["doc_id"]
        .nunique()
        .rename("size")
    )
    entity_node_rows = [
        {
            "Id":          f"ent_{slugify(c)}",
            "Label":       c,
            "node_type":   "entity",
            "entity_type": sheet_type,
            "size":        int(entity_doc_freq.get(c, 0)),
        }
        for c in sorted(canon_sets[sheet_type])
        if c in entity_doc_freq
    ]
    nodes_df = pd.concat(
        [src_nodes_df, pd.DataFrame(entity_node_rows)],
        ignore_index=True,
    )

    # collision check
    dupes = nodes_df[nodes_df.duplicated("Id", keep=False)]
    if len(dupes):
        print(f"  WARNING ({sheet_type}): duplicate node Ids: {dupes['Id'].tolist()}")

    entity_id = {
        row["Label"]: row["Id"]
        for _, row in nodes_df[nodes_df["node_type"] == "entity"].iterrows()
    }

    # --- edges ---
    mention_counts = (
        type_long
        .groupby(["arena", "subunit", "canonical"])["doc_id"]
        .nunique()
        .reset_index(name="n")
    )

    edge_rows = []
    for _, row in mention_counts.iterrows():
        key = (row["arena"], row["subunit"])
        if key not in subunit_totals:
            continue
        src = subunit_id.get(row["subunit"])
        tgt = entity_id.get(row["canonical"])
        if src is None or tgt is None:
            continue
        weight = row["n"] / subunit_totals[key]
        if weight < MIN_EDGE_WEIGHT:
            continue
        edge_rows.append({
            "Source": src,
            "Target": tgt,
            "Weight": round(weight, 6),
            "Type":   "Undirected",
        })

    edges_df = pd.DataFrame(edge_rows)

    # --- export ---
    nodes_out = NB_DIR / f"{prefix}_nodes.csv"
    edges_out = NB_DIR / f"{prefix}_edges.csv"
    nodes_df.to_csv(nodes_out, index=False)
    edges_df.to_csv(edges_out, index=False)

    # --- summary ---
    print(f"{sheet_type}:")
    print(f"  Entity nodes : {len(entity_node_rows)}")
    print(f"  Source nodes : {len(src_nodes_df)}")
    print(f"  Edges        : {len(edges_df)}")
    print(f"  → {nodes_out.name}, {edges_out.name}")
    print()

print("=" * 60)

PER:
  Entity nodes : 60
  Source nodes : 20
  Edges        : 513
  → per_nodes.csv, per_edges.csv

ORG:
  Entity nodes : 80
  Source nodes : 20
  Edges        : 845
  → org_nodes.csv, org_edges.csv

LOC:
  Entity nodes : 38
  Source nodes : 20
  Edges        : 465
  → loc_nodes.csv, loc_edges.csv

